# CodeAlpha Machine Learning Internship
## Task 3: Handwritten Character Recognition

### 🎯 Objective
Identify handwritten characters and alphabets (A-Z) from 28x28 grayscale images using image processing and deep learning architectures.

### 🛠️ Key Pipeline Components
- **Dataset**: A-Z Handwritten Alphabets / EMNIST format
- **Image Processing**: Grayscale conversion, adaptive thresholding, bounding box cropping, and center-of-mass alignment
- **Deep Learning Model**: Multi-layer Convolutional/Deep Neural Network with ReLU activations, Adam optimizer, and Softmax classification
- **Evaluation**: Accuracy, Precision, Recall, Macro F1-Score, and 26x26 Confusion Matrix Heatmap

### Step 1: Environment Setup & Library Imports

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
PROJECT_ROOT = Path(".").resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.models import DeepCharacterClassifier
from src.image_processing import preprocess_image_array, array_to_image_preview
from src.evaluate import calculate_character_metrics, plot_character_confusion_matrix, plot_loss_curve

CLASS_LABELS = [chr(i) for i in range(ord('A'), ord('Z') + 1)]
print("Environment initialized successfully!")

### Step 2: Load the Character Dataset

In [ ]:
data_path = PROJECT_ROOT / "data" / "raw" / "characters_data.csv"

if not data_path.exists():
    print("Dataset missing locally. Generating or downloading via KaggleHub...")
    from src.download_data import download_from_kagglehub
    download_from_kagglehub()

df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape} (Records x Pixels)")
print(f"Target Classes: {len(np.unique(df.iloc[:, 0]))} unique alphabets (A-Z)")
df.head()

### Step 3: Visualize Sample Handwritten Characters

In [ ]:
plt.figure(figsize=(10, 6))
for i in range(15):
    plt.subplot(3, 5, i + 1)
    row = df.iloc[i]
    label = int(row.iloc[0])
    pixels = row.iloc[1:].values.astype(float).reshape(28, 28)
    plt.imshow(pixels, cmap="gray")
    plt.title(f"Char: '{CLASS_LABELS[label]}'", fontsize=11)
    plt.axis("off")
plt.suptitle("Sample Handwritten Characters from Dataset", fontsize=13)
plt.tight_layout()
plt.show()

### Step 4: Prepare Train & Test Sets

In [ ]:
from sklearn.model_selection import train_test_split

y = df.iloc[:, 0].astype(int).values
X = df.iloc[:, 1:].values.astype(np.float32)

# Normalize to [0, 1]
if np.max(X) > 1.0:
    X = X / 255.0

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train samples: {len(X_train)}")
print(f"Test samples : {len(X_test)}")

### Step 5: Define & Train Deep Neural Network
Architecture: 784 Inputs $\rightarrow$ Dense(256, ReLU) $\rightarrow$ Dense(128, ReLU) $\rightarrow$ Dense(64, ReLU) $\rightarrow$ Dense(26, Softmax).

In [ ]:
clf = DeepCharacterClassifier(
    hidden_layer_sizes=(256, 128, 64),
    activation="relu",
    solver="adam",
    batch_size=64,
    learning_rate_init=0.001,
    max_iter=100,
    random_state=42,
    early_stopping=True
)

print("Training Deep Character Network...")
clf.fit(X_train, y_train)
print("Training complete!")

### Step 6: Loss Convergence Curve

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(clf.loss_curve_, color="#2563EB", lw=2)
plt.title("Deep Learning Training Loss Reduction over Epochs", fontsize=13)
plt.xlabel("Iterations (Epochs)")
plt.ylabel("Cross-Entropy Loss")
plt.grid(True, linestyle=":", alpha=0.6)
plt.show()

### Step 7: Model Evaluation Metrics & Classification Report

In [ ]:
metrics = calculate_character_metrics(clf, X_test, y_test)
print(f"Test Accuracy    : {metrics['Accuracy'] * 100:.2f}%")
print(f"Precision (Macro): {metrics['Precision (Macro)']:.4f}")
print(f"Recall (Macro)   : {metrics['Recall (Macro)']:.4f}")
print(f"F1-Score (Macro) : {metrics['F1-Score (Macro)']:.4f}")
print("\nClassification Report:")
print(metrics["Classification Report"])

### Step 8: Multi-Class 26x26 Confusion Matrix Heatmap

In [ ]:
cm = metrics["Confusion Matrix"]
cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(10, 8))
sns.heatmap(cm_norm, cmap="Blues", xticklabels=CLASS_LABELS, yticklabels=CLASS_LABELS)
plt.title("Confusion Matrix Heatmap (A-Z Alphabets)", fontsize=13)
plt.xlabel("Predicted Character")
plt.ylabel("True Character")
plt.show()

### Step 9: Sample Prediction Grid (True vs. Predicted)

In [ ]:
from src.evaluate import plot_sample_predictions
y_pred = clf.predict(X_test)

plt.figure(figsize=(10, 8))
sample_indices = np.random.choice(len(y_test), 12, replace=False)
for i, idx in enumerate(sample_indices):
    plt.subplot(3, 4, i + 1)
    img = X_test[idx].reshape(28, 28)
    t_char = CLASS_LABELS[y_test[idx]]
    p_char = CLASS_LABELS[y_pred[idx]]
    col = "green" if t_char == p_char else "red"
    plt.imshow(img, cmap="gray")
    plt.title(f"True: {t_char} | Pred: {p_char}", color=col, fontsize=11, fontweight="bold")
    plt.axis("off")
plt.suptitle("Sample Predictions on Test Set", fontsize=13)
plt.tight_layout()
plt.show()

### Step 10: Conclusion & Deliverables
1. **Deep Learning Accuracy**: The multi-layer deep network successfully learned non-linear stroke features for all 26 alphabets.
2. **Artifacts Exported**: Trained weights saved to `models/handwritten_character_model.joblib`.
3. **Interactive Demo**: The Streamlit canvas app in `visualizer/app.py` allows interactive mouse/touchpad drawing and real-time recognition!